# SEC Fundamentals Daily Refresh
Incremental ingestion of SEC EDGAR filings and earnings calendar.

**Outputs**:
- `{catalog}.gold.sec_fundamentals` -- 10-K, 10-Q filings with financials
- `{catalog}.gold.earnings_calendar` -- upcoming/recent earnings dates + EPS

**Sources**: SEC EDGAR API (free, User-Agent required) + yfinance earnings

**Schedule**: Daily 8 PM ET

**Parameters**:
| Widget | Default | Description |
|--------|---------|-------------|
| `catalog` | riskbricks | Unity Catalog name |
| `as_of_date` | (auto) | Yesterday ET |
| `max_symbols` | 0 | 0=all from company_universe |
| `edgar_email` | riskbricks@example.com | SEC EDGAR User-Agent email |

In [0]:
%pip install yfinance --quiet
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog", "riskbricks", "Catalog Name")
dbutils.widgets.text("as_of_date", "", "As of Date (YYYY-MM-DD, blank=yesterday)")
dbutils.widgets.text("max_symbols", "0", "Max Symbols (0=all)")
dbutils.widgets.text("edgar_email", "riskbricks@example.com", "SEC EDGAR User-Agent Email")

In [0]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import time
import json
import requests
import pandas as pd
import yfinance as yf
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType,
)

# -- Read widgets -------------------------------------------------------
catalog = dbutils.widgets.get("catalog").strip()
local_tz = ZoneInfo("America/New_York")
edgar_email = dbutils.widgets.get("edgar_email").strip()

as_of_input = dbutils.widgets.get("as_of_date").strip()
if as_of_input:
    as_of_date = as_of_input
else:
    as_of_date = (datetime.now(local_tz).date() - timedelta(days=1)).strftime("%Y-%m-%d")

max_symbols = int(dbutils.widgets.get("max_symbols") or "0")

# -- Ensure schemas exist -----------------------------------------------
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

# -- Load symbols from company_universe ---------------------------------
universe_df = spark.sql(f"""
    SELECT DISTINCT symbol, company_name
    FROM {catalog}.gold.company_universe
    ORDER BY symbol
""").collect()

symbols = [row.symbol for row in universe_df]
symbol_to_name = {row.symbol: row.company_name for row in universe_df}
if max_symbols and max_symbols > 0:
    symbols = symbols[:max_symbols]

# SEC EDGAR needs CIK lookup -- build via SEC company tickers JSON
EDGAR_HEADERS = {"User-Agent": f"RiskBricks DataPlatform {edgar_email}"}

print(f"Config: catalog={catalog}, as_of_date={as_of_date}")
print(f"Symbols: {len(symbols)}, EDGAR email: {edgar_email}")

In [0]:
def get_cik_map(symbols, headers):
    """Build symbol -> CIK mapping from SEC EDGAR company tickers."""
    url = "https://www.sec.gov/files/company_tickers.json"
    try:
        resp = requests.get(url, headers=headers, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        # data is {"0": {"cik_str": 320193, "ticker": "AAPL", ...}, ...}
        cik_map = {}
        for entry in data.values():
            ticker = entry.get("ticker", "").upper()
            cik = entry.get("cik_str")
            if ticker in symbols and cik:
                cik_map[ticker] = str(cik).zfill(10)  # SEC uses 10-digit padded CIK
        return cik_map
    except Exception as e:
        print(f"Failed to load CIK map: {e}")
        return {}

def get_company_facts(cik, headers):
    """Fetch company facts from SEC EDGAR XBRL API."""
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    try:
        resp = requests.get(url, headers=headers, timeout=30)
        if resp.status_code == 404:
            return None
        resp.raise_for_status()
        return resp.json()
    except Exception:
        return None

cik_map = get_cik_map(symbols, EDGAR_HEADERS)
print(f"CIK mapping found for {len(cik_map)}/{len(symbols)} symbols")

sec_rows = []
for sym in symbols:
    cik = cik_map.get(sym)
    if not cik:
        continue
    facts = get_company_facts(cik, EDGAR_HEADERS)
    if not facts:
        continue

    us_gaap = facts.get("facts", {}).get("us-gaap", {})
    # Extract key financial metrics
    metrics = {
        "Revenue": us_gaap.get("Revenues", us_gaap.get("RevenueFromContractWithCustomerExcludingAssessedTax", {})),
        "NetIncome": us_gaap.get("NetIncomeLoss", {}),
        "TotalAssets": us_gaap.get("Assets", {}),
        "TotalDebt": us_gaap.get("LongTermDebt", us_gaap.get("LongTermDebtNoncurrent", {})),
        "EPS": us_gaap.get("EarningsPerShareDiluted", us_gaap.get("EarningsPerShareBasic", {})),
    }

    for metric_name, metric_data in metrics.items():
        units = metric_data.get("units", {})
        for unit_key, entries in units.items():
            # Get only the most recent filings (last 8)
            recent = sorted(entries, key=lambda x: x.get("filed", ""), reverse=True)[:8]
            for entry in recent:
                filed = entry.get("filed")
                if filed and filed >= "2025-01-01":  # only recent
                    sec_rows.append({
                        "symbol": sym,
                        "company_name": symbol_to_name.get(sym, sym),
                        "cik": cik,
                        "metric": metric_name,
                        "value": float(entry.get("val", 0)),
                        "unit": unit_key,
                        "period_end": entry.get("end"),
                        "filed_date": filed,
                        "form_type": entry.get("form", ""),
                        "fiscal_year": str(entry.get("fy", "")),
                        "fiscal_period": str(entry.get("fp", "")),
                        "as_of_date": as_of_date,
                    })
    time.sleep(0.15)  # SEC rate limit: 10 req/sec

print(f"SEC filings: {len(sec_rows)} rows from {len(cik_map)} companies")

In [0]:
earnings_rows = []
for i, sym in enumerate(symbols):
    try:
        t = yf.Ticker(sym)
        dates = t.get_earnings_dates(limit=8)
        if dates is not None and not dates.empty:
            dates = dates.reset_index()
            for _, row in dates.iterrows():
                ed = row.get("Earnings Date")
                if pd.isna(ed):
                    continue
                earnings_rows.append({
                    "symbol": sym,
                    "event_date": ed.to_pydatetime() if hasattr(ed, "to_pydatetime") else None,
                    "eps_estimate": float(row["EPS Estimate"]) if pd.notna(row.get("EPS Estimate")) else None,
                    "eps_actual": float(row["Reported EPS"]) if pd.notna(row.get("Reported EPS")) else None,
                    "surprise_pct": float(row["Surprise(%)"]) if pd.notna(row.get("Surprise(%)")) else None,
                    "as_of_date": as_of_date,
                })
    except Exception:
        pass
    if (i + 1) % 50 == 0:
        print(f"  Earnings progress: {i+1}/{len(symbols)}")
    time.sleep(0.15)

print(f"Earnings calendar: {len(earnings_rows)} rows")

In [0]:
def write_gold_table(table_name, rows, schema):
    if not rows:
        print(f"  No rows for {table_name}")
        return 0
    df = spark.createDataFrame(rows, schema=schema)
    df = df.withColumn("ingestion_timestamp", F.current_timestamp())
    if not spark.catalog.tableExists(table_name):
        df.write.mode("overwrite").partitionBy("as_of_date", "symbol").saveAsTable(table_name)
    else:
        spark.sql(f"DELETE FROM {table_name} WHERE as_of_date = '{as_of_date}'")
        df.write.mode("append").saveAsTable(table_name)
    cnt = df.count()
    print(f"  Saved {cnt} rows to {table_name}")
    return cnt

sec_schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("company_name", StringType(), True),
    StructField("cik", StringType(), True),
    StructField("metric", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("unit", StringType(), True),
    StructField("period_end", StringType(), True),
    StructField("filed_date", StringType(), True),
    StructField("form_type", StringType(), True),
    StructField("fiscal_year", StringType(), True),
    StructField("fiscal_period", StringType(), True),
    StructField("as_of_date", StringType(), False),
])

earnings_schema = StructType([
    StructField("symbol", StringType(), False),
    StructField("event_date", TimestampType(), True),
    StructField("eps_estimate", DoubleType(), True),
    StructField("eps_actual", DoubleType(), True),
    StructField("surprise_pct", DoubleType(), True),
    StructField("as_of_date", StringType(), False),
])

sec_cnt = write_gold_table(f"{catalog}.gold.sec_fundamentals", sec_rows, sec_schema)
earn_cnt = write_gold_table(f"{catalog}.gold.earnings_calendar", earnings_rows, earnings_schema)

In [0]:
result = {
    "status": "success",
    "catalog": catalog,
    "as_of_date": as_of_date,
    "sec_fundamentals": sec_cnt,
    "earnings_calendar": earn_cnt,
    "symbols": len(symbols),
    "cik_mapped": len(cik_map),
}

print("=" * 60)
print("SEC FUNDAMENTALS DAILY REFRESH COMPLETE")
print("=" * 60)
print(f"  Catalog:          {catalog}")
print(f"  As of date:       {as_of_date}")
print(f"  CIK mapped:       {len(cik_map)}/{len(symbols)}")
print(f"  SEC filings:      {sec_cnt:,}")
print(f"  Earnings cal:     {earn_cnt:,}")

dbutils.notebook.exit(json.dumps(result))